In [19]:
import random
import torch
import wandb
import time
import os
from tqdm import tqdm
import numpy as np
import pandas as pd
from random import choices
# import matplotlib.pyplot as plt
from lion_pytorch import Lion

import os, torch, logging
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from huggingface_hub import login
# from flash_attn import flash_attn_func
import accelerate
import wandb
from tqdm import tqdm
import trl

tqdm.pandas()

from datasets import load_dataset

from transformers import AutoTokenizer, pipeline

from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead, create_reference_model

### Configuration

In [20]:
sentiment_pipe_kwargs = {"top_k": None, "function_to_apply": "none"}

config = PPOConfig(
	gradient_accumulation_steps=1,
	learning_rate=1.41e-5,
	# max_grad_norm=wandb.config["max_grad_norm"],
	log_with="wandb",
	optimize_cuda_cache=True,
	optimize_device_cache=True,
	early_stopping=True,
	is_peft_model=True,
	use_score_scaling=True,
	use_score_norm=True,
 	# dtype=torch.float16
	# score_clip=wandb.config["score_clip"],
)

txt_in_len = 100
txt_out_len = 100
seed = 1

In [21]:
np.random.seed(seed)

In [5]:
from unsloth import FastLanguageModel
dtype=None

model, llama_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-bnb-4bit", # Supports Llama, Mistral - replace this!
    max_seq_length = 1024, # Supports RoPE Scaling internally, so choose any!
    # dtype = None,
    use_cache=False,
    pretraining_tp=1,
    device_map="auto",
    load_in_4bit = True,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
    dtype=dtype #None for auto
)
# Do model patching and add fast LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    use_rslora = True,
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    # dype=dtype
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


==((====))==  Unsloth: Fast Llama patching release 2024.5
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.669 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. Xformers = 0.0.26.post1. FA = True.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


Unsloth 2024.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [6]:
model.print_trainable_parameters()

trainable params: 39,976,960 || all params: 6,778,392,576 || trainable%: 0.589770503135875


In [7]:
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"  # Fix for fp16

In [8]:
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    model,
    # torch_dtype=torch.float16,
    device_map="auto",
    is_trainable=True, 
    # peft_config= lora_config
    torch_dtype=torch.float16
    )

In [9]:
# # create the dataset
# #
# dataset = load_dataset("imdb", split="train[:10%]")
# dataset = dataset.rename_columns({"text": "review", "label": "sentiment"})
# # make sure the comments are are at least 500 and trim to 1000
# dataset = dataset.filter(lambda x: len(x["review"]) > 500, batched=False)
# dataset = dataset.map(lambda x: {"review": x["review"][:1000]}, batched=False)

# dataset

Dataset({
    features: ['review', 'sentiment'],
    num_rows: 2261
})

In [57]:
def preprocessing(data):
    # data = data.rename_column("utterance", "text")
    # data = data.rename_column("emotion", "label")
    data = data.rename_columns({"utterance": "review", "emotion": "sentiment"})
    data = data.remove_columns(["dialog_id", "turn_type"])
    # data = data.filter(lambda x: len(x["review"]) > 500, batched=False)
    # data = data.map(lambda x: {"review": x["review"][:1000]}, batched=False)
    return data

In [58]:
data_name = "benjaminbeilharz/better_daily_dialog"
data_raw = load_dataset(data_name, num_proc=16, split="train[:5%]")
data_raw = preprocessing(data_raw)
dataset = data_raw
dataset

Dataset({
    features: ['review', 'sentiment'],
    num_rows: 4358
})

### Tokenize IMDB reviews

We tokenize all IMDB in advance to avoid tokenizing twice. In the first step we encode the queries and slice the first `txt_in_len` tokens. In a second step we decode these tokens back to text for later display.

In [59]:
dataset = dataset.map(
    lambda x: {"input_ids": llama_tokenizer.encode(" " + x["review"], return_tensors="pt")[0, :txt_in_len]},
    batched=False,
)
dataset = dataset.map(lambda x: {"query": llama_tokenizer.decode(x["input_ids"])}, batched=False)
dataset = dataset[:20480]

from datasets import Dataset

dataset = Dataset.from_dict(dataset)
dataset.set_format("pytorch")

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

In [60]:
dataset[3]["input_ids"]

tensor([   1,  259, 1938,  366, 2289])

In [61]:
def collator(data):
    return dict((key, [d[key] for d in data]) for key in data[0])

In [62]:
optimizer = Lion(filter(lambda p: p.requires_grad, ppo_model.parameters()), lr=config.learning_rate)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

In [63]:
ppo_trainer = PPOTrainer(
                            config = config,
                            model = ppo_model,
                            tokenizer = llama_tokenizer,
                            dataset = dataset,
                            data_collator = collator,
                            optimizer = optimizer,
                            lr_scheduler = lr_scheduler
)

In [64]:
ppo_trainer

### Load BERT classifier
We load a BERT classifier fine-tuned on the IMDB dataset.

In [65]:
# ppo_trainer.model.gradient_checkpointing_enable()

In [66]:
if ppo_trainer.accelerator.num_processes == 1:
    device = 0 if torch.cuda.is_available() else "cpu"  # to avoid a `pipeline` bug
else:
    device = ppo_trainer.accelerator.device
# sentiment_pipe = pipeline("sentiment-analysis", "lvwerra/distilbert-imdb", device=device)

In [67]:
sentiment_pipe = pipeline("sentiment-analysis", "/home/user/github/chat-bot/SA-v2", device=device)

The model outputs are the logits for the negative and positive class. We will use the logits for positive class as a reward signal for the language model.

In [68]:
text = "this movie was really bad!!"
output = sentiment_pipe(text, **sentiment_pipe_kwargs)
output

[{'label': 'anger', 'score': 2.773132562637329},
 {'label': 'sadness', 'score': 2.5415053367614746},
 {'label': 'fear', 'score': 0.05701369047164917},
 {'label': 'joy', 'score': -0.4793158173561096},
 {'label': 'surprise', 'score': -0.5914648771286011},
 {'label': 'disgust', 'score': -0.6156034469604492},
 {'label': 'neutral', 'score': -3.191633939743042}]

In [69]:
text = "this movie was really good!!"
output = sentiment_pipe(text, **sentiment_pipe_kwargs)
output

[{'label': 'joy', 'score': 5.5541486740112305},
 {'label': 'surprise', 'score': 1.870810866355896},
 {'label': 'anger', 'score': -0.1293402463197708},
 {'label': 'neutral', 'score': -0.312915563583374},
 {'label': 'fear', 'score': -1.5887203216552734},
 {'label': 'sadness', 'score': -1.81271231174469},
 {'label': 'disgust', 'score': -2.2504804134368896}]

In [70]:
text = "this movie was a documentary"
output = sentiment_pipe(text, **sentiment_pipe_kwargs)
output

[{'label': 'neutral', 'score': 5.421863555908203},
 {'label': 'surprise', 'score': 0.7959042191505432},
 {'label': 'joy', 'score': -0.7838171124458313},
 {'label': 'sadness', 'score': -0.89622962474823},
 {'label': 'anger', 'score': -0.9333758354187012},
 {'label': 'fear', 'score': -1.8126181364059448},
 {'label': 'disgust', 'score': -1.8816487789154053}]

The resulting reward signal:

In [71]:
def extract_pipe_output(outputs):
    positive_logits = []
    for out in outputs:
        for element in out:
            if element["label"] == "joy":
                positive_logits.append(torch.tensor(element["score"]))
    return positive_logits

In [72]:
output

[{'label': 'neutral', 'score': 5.421863555908203},
 {'label': 'surprise', 'score': 0.7959042191505432},
 {'label': 'joy', 'score': -0.7838171124458313},
 {'label': 'sadness', 'score': -0.89622962474823},
 {'label': 'anger', 'score': -0.9333758354187012},
 {'label': 'fear', 'score': -1.8126181364059448},
 {'label': 'disgust', 'score': -1.8816487789154053}]

### Control token dict
We will append the control token at the beginning of each query to signal the model what the target sentiment is. Each control sequence consists of three tokens:

In [73]:
ctrl_str = ["[neutral]", "[surprise]", "[joy]", "[sadness]", "[anger]", "[fear]", "[disgust]"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # this should be handled by accelerate
ctrl_tokens = dict((s, llama_tokenizer.encode(s, return_tensors="pt").squeeze().to(device)) for s in ctrl_str)

In [74]:
ctrl_tokens

{'[neutral]': tensor([    1,   518, 17821,  1705, 29962], device='cuda:0'),
 '[surprise]': tensor([    1,   518,  7610,  7734, 29962], device='cuda:0'),
 '[joy]': tensor([    1,   518,  2212, 29891, 29962], device='cuda:0'),
 '[sadness]': tensor([    1,   518, 29879,   328,  2264, 29962], device='cuda:0'),
 '[anger]': tensor([    1,   518,  4600, 29962], device='cuda:0'),
 '[fear]': tensor([    1,   518, 29888,   799, 29962], device='cuda:0'),
 '[disgust]': tensor([    1,   518,  2218, 29887,   504, 29962], device='cuda:0')}

### Reward function

In [75]:
# def pos_logit_to_reward(logit, task):
#     """
#     Take the positive sentiment logit and scale it for the task.
#         task [negative]: reward = -logit
#         task [neutral]: reward = -2*abs(logit)+4
#         task [positive]: reward = logit
#     """
#     for i in range(len(logit)):
#         if task[i] == "[negative]":
#             logit[i] = -logit[i]
#         elif task[i] == "[neutral]":
#             logit[i] = -2 * torch.abs(logit[i]) + 4
#         elif task[i] == "[positive]":
#             pass
#         else:
#             raise ValueError("task has to be in [0, 1, 2]!")
#     return logit

In [76]:
def pos_logit_to_reward(logit, task):
    """
    Take the positive sentiment logit and scale it for the task.
        task [neutral]: reward = -2 * abs(logit) + 4
        task [surprise]: reward = 2 * logit
        task [joy]: reward = logit
        task [sadness]: reward = -logit
        task [anger]: reward = -1.5 * logit
        task [fear]: reward = -1.2 * logit
        task [disgust]: reward = -logit
    """
    for i in range(len(logit)):
        if task[i] == "[neutral]":
            logit[i] = -2 * torch.abs(logit[i]) + 4
        elif task[i] == "[surprise]":
            logit[i] = 2 * logit[i]
        elif task[i] == "[joy]":
            pass  # logit[i] = logit[i] 這行可以省略，因為不變
        elif task[i] == "[sadness]":
            logit[i] = -logit[i]
        elif task[i] == "[anger]":
            logit[i] = -1.5 * logit[i]
        elif task[i] == "[fear]":
            logit[i] = -1.2 * logit[i]
        elif task[i] == "[disgust]":
            logit[i] = -logit[i]
        else:
            raise ValueError("Invalid task label!")
    return logit


The following examples show the rewards for the cases where the classifier logit is 4, -4 and 0 for the three targets `['negative]`, `['neutral]` and `['positive']`. The scaling is not perfect as it differs between neutral and the other two classes. This is something to further investigate in the future. Ideally, one would use the logit output for each class individually, but since there is no dedicated class for neutral this is a workaround.

In [77]:
print(ctrl_str)

['[neutral]', '[surprise]', '[joy]', '[sadness]', '[anger]', '[fear]', '[disgust]']


In [78]:
pos_logit_to_reward(torch.Tensor([4, 4, 4, 4, 4, 4, 4]), ctrl_str)

tensor([-4.0000,  8.0000,  4.0000, -4.0000, -6.0000, -4.8000, -4.0000])

In [79]:
pos_logit_to_reward(torch.Tensor([-4, -4, -4, 4, -4, 4]), ctrl_str)

tensor([-4.0000, -8.0000, -4.0000, -4.0000,  6.0000, -4.8000])

In [80]:
pos_logit_to_reward(torch.Tensor([0, 0, 0, 0, 0, 0, 0]), ctrl_str)

tensor([4., 0., 0., -0., -0., -0., -0.])

### Generation settings

In [81]:
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": llama_tokenizer.eos_token_id,
    "max_new_tokens": txt_out_len,
    "eos_token_id": -1,
}

## Optimize model

**Steps**

The training loop consists of the following steps:
1. Get a batch of queries and create random controls
2. Get the query responses from the policy
3. Join query and responses and tokenize for BERT analysis
4. Get sentiments for query/responses from BERT
5. Optimize policy with PPO using the (query, response, reward) triplet
6. Log all the training statistics

**Training time**

This step takes **~2h** on a P6000 GPU with the above specified settings.

[RuntimeError: expected mat1 and mat2 to have the same dtype, but got: c10::BFloat16 != float](https://github.com/unslothai/unsloth/issues/222#issuecomment-2005708803)

In [82]:
trl.trainer.peft_module_casting_to_bf16(ppo_model)

In [83]:
for epoch in range(2):
    for batch in tqdm(ppo_trainer.dataloader):
        (logs, game_data,) = (
            dict(),
            dict(),
        )

        #### prepend a random control token
        task_list = choices(ctrl_str, k=config.batch_size)
        game_data["query"] = [t + q for t, q in zip(task_list, batch["query"])]
        query_tensors = [torch.cat((ctrl_tokens[t], input_ids)) for t, input_ids in zip(task_list, batch["input_ids"])]
        # print("prepend a random control token: ", torch.cuda.memory_allocated())
        
        #### get response from gpt2
        response_tensors = []
        for query in query_tensors:
            response = ppo_trainer.generate(query, **generation_kwargs)
            response_tensors.append(response.squeeze()[-txt_out_len:])
        game_data["response"] = [llama_tokenizer.decode(r.squeeze()) for r in response_tensors]
        # print("get response from gpt2: ", torch.cuda.memory_allocated())

        #### sentiment analysis
        texts = [q + r for q, r in zip(batch["query"], game_data["response"])]
        logits = extract_pipe_output(sentiment_pipe(texts, **sentiment_pipe_kwargs))
        rewards = pos_logit_to_reward(logits, task_list)
        # print("sentiment analysis: ", torch.cuda.memory_allocated())
        #### Run PPO training
        t = time.time()
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        # print("Run PPO training: ", torch.cuda.memory_allocated())

        for cs in ctrl_str:
            key = "env/reward_" + cs.strip("[]")
            stats[key] = np.mean([r.cpu().numpy() for r, t in zip(rewards, task_list) if t == cs])
            # print("inside: ", torch.cuda.memory_allocated())
        ppo_trainer.log_stats(stats, game_data, rewards)

  0%|          | 0/34 [00:00<?, ?it/s]

 18%|█▊        | 6/34 [17:21<1:20:58, 173.52s/it]


KeyboardInterrupt: 

### Training progress
If you are tracking the training progress with Weights&Biases you should see a plot similar to the following:

<div style="text-align: center">
<img src='https://huggingface.co/datasets/trl-internal-testing/example-images/resolve/main/images/gpt2-ctrl-training-stats.png' width='800'>
<p style="text-align: center;"> <b>Figure:</b> Reward mean and distribution evolution during training. </p>
</div>

One can observe how the model starts to generate more positive outputs after a few optimisation steps.

> Note: Investigating the KL-divergence will probably show that at this point the model has not converged to the target KL-divergence, yet. To get there would require longer training or starting with a higher inital coefficient.

## Model inspection

### Reward distribution
First, we can have a look at the reward distribution. Both the negative and positive rewards are clearly shifted to high rewards. The neutral rewards, however, are still centered around zero. There are a few possible explanations for this. There could be a bug in the code and the way the neutral rewards are calculated. Another problem could be that sentence sometimes start with a strong sentiment and it is hard for the model shift the sentiment towards neutral.

In [ ]:
# for ctrl_s in ctrl_str:
#     plt.hist(
#         [r for r, t in zip(logs["env/reward_dist"], task_list) if t == ctrl_s], density=True, alpha=0.5, label=ctrl_s
#     )
# plt.legend(loc="best")
# plt.title("reward distribution")
# plt.grid(True)
# plt.show()

## Save model
Finally, we save the model to disk for later usage.

In [ ]:
ppo_model.save_pretrained("llama-sntm-ctrl")
llama_tokenizer.save_pretrained("llama-sntm-ctrl")

In [ ]:
MODEL_PATH = "./llama-sntm-ctrl"
pipe = pipeline("text-generation", model=MODEL_PATH, 
                tokenizer=MODEL_PATH, max_length=100, 
                num_return_sequences=5)

In [ ]:
pipe("[joy]What are we going to do in the morning?", do_sample=False)